# 第一章：市场、资产与交易对象

一张价格表并不是一个可以直接回测的市场。表中的 100 可能是每份基金的成交价，也可能是期货点位；表中的 4 可能是债券收益率的百分数，也可能是期权权利金。即使同样写着“收盘价”，它们对账户现金、持仓数量和下一天损益的影响也不同。

本章只追踪一个问题：**一个初始有 1000 元的账户，买入一只基金并持有几天，为什么赚钱或亏钱？** 我们先搭出只有现金和持仓的账本，再加入报价、分红、拆股、合同乘数和交易时间。股票、债券、外汇、期货和期权都沿着这个问题展开：如果把账本中的基金替换成这种合同，哪一列必须改变？这样读完后，资产名称会对应到可计算的现金流，而不只是词汇。

## 0. 学习目标与研究任务

读完本章，你应当能把一个交易对象说明白：持有什么权利，报价单位是什么，持有一份需要多少资金，哪些事件改变数量和现金，数据何时可见，订单何时可能成交。进一步，你应当能解释为什么股息不等于凭空赚钱、拆股不等于亏损一半、债券收益率上升不等于债券上涨、期货换月价差不等于当场亏损。

本章的最简实现是一个逐日账户账本。它默认只读本地文件，教学行情由代码直接给出；无需账户、网络或密钥。我们暂不预测价格，也不寻找最优策略：连“赚了多少钱”都没有定义清楚时，预测准确率没有可解释的经济意义。读者可以把数学注意力放在现金流和合同上，不必重新学习基础概率统计。


## 1. 先把研究问题写成一笔持仓

设一只教学基金名为 `E`，每份当前价格 100 元。账户有 1000 元，买入 6 份后，持仓市值为 600 元，剩余现金 400 元。第二天价格变成 102 元，没有交易和分红，账户价值是 \(400+6\times102=1012\) 元。价格涨了 2%，但账户只涨了 1.2%，因为只有 60% 的资金参与了这次价格变化。

这里已经出现三个必须分开的量。**价格（price）** 是每份资产的报价，单位是“元/份”；**持仓数量（quantity）** 是账户拥有多少份；**持仓市值（market value）** 是数量乘价格，单位是元。一个模型可以对价格变化做出预测，真正的损益却还取决于买了多少以及现金如何处理。

先不考虑分红应收款，账户净值可写成

\[
V_t=B_t+q_tP_t.
\]

\(V_t\) 是第 \(t\) 个观察时点的账户总价值，\(B_t\) 是现金余额，\(q_t\) 是持有份数，\(P_t\) 是每份的估值价格。这里“净值”指账户总财富；基金公布的每份净值是另一种口径，后面会区分。如果账户有负债，还要从右侧扣除负债，而不能把借来的资金也当作自己的盈利。

这笔交易的**标的（underlying）** 是基金份额对应的一篮子股票；**名义规模（notional exposure）** 是 600 元的股票市场敞口。敞口指某种价格变动会影响多少经济价值。现金支付、名义规模和最大可能损失在这笔普通现货交易中很接近，在期货和期权中则会分开。

现实中研究者拿到的列未必是可持有的基金。下面三个公开市场指标正好说明问题：股票指数是一个组合规则算出来的数，债券收益率是折现参数，波动率指标则来自期权报价。它们都可用于解释市场，但不能不加处理地塞进 \(qP\) 当作账户资产。




![三类单位不同的市场对象：股票指数、收益率报价、隐含波动率指标](figures/01_market_objects.png)

**图 1-1｜三种市场对象，三种单位。** 同一张图里的三条曲线单位完全不同：S&P 500 是价格指数点位，^TNX 是 10 年期美债收益率报价，VIX 是期权隐含波动率指标。把它们画在一起是为了提醒：收益率上升不等于债券价格上涨，VIX 也不是可以按“现货价”直接长期持有的资产。读图时把每一列补成数据字典：来源、单位、时间戳、何时可见、是否可交易。数据类型：公开数据本地快照（Yahoo Finance ^GSPC/^TNX 与公开 VIX 日线），由本地插图生成器绘制。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# 确定性教学数据：不对应任何真实基金。
dates = pd.bdate_range("2024-01-08", periods=5)
events = pd.DataFrame({
    "raw_close": [100.0, 102.0, 101.0, 50.5, 52.0],
    "split_ratio": [1.0, 1.0, 1.0, 2.0, 1.0],
    "dividend_per_share": [0.0, 0.0, 1.0, 0.0, 0.0],
    "dividend_paid": [0.0, 0.0, 0.0, 0.0, 6.0],
}, index=dates)
# 先只看没有公司行动的前两天：买入发生在第一天估值之前。
plain_ledger = events.iloc[:2][["raw_close"]].copy()
plain_ledger["quantity"] = 6.0
plain_ledger["cash"] = 1000.0 - 6.0 * 100.0
plain_ledger["account_value"] = plain_ledger.cash + plain_ledger.quantity * plain_ledger.raw_close
plain_ledger


代码故意只算前两天。第三天和第四天分别有分红和拆股；如果仍然把“价格乘 6”复制下去，就会制造错误的损益。我们会在理解合同之后补齐账本。研究中一个很有用的习惯是：先写出哪几行暂时不能解释，再扩展模型；不要让一列数值因为没有报错就自动获得经济含义。

### 1.1 市场价格为什么变化

把 `E` 想成一家公司的股票组合。公司未来盈利预期提高，投资者可能愿意付更多钱；即使盈利预期不变，要求的回报率下降也会提高今天的估值。最简单的一期折现是 \(P_0=E_0[C_1]/(1+k)\)：\(P_0\) 是现价，\(C_1\) 是一期后分给持有者的现金及终值，\(E_0\) 表示基于当前信息的预期，\(k\) 是要求的回报率。若预期收入为 110，要求回报 10%，现值为 100；要求回报降到 5%，同一现金流现值变为约 104.76。

这就是**风险溢价（risk premium）** 的入口：承担不确定性的人通常要求比近似无风险现金更高的回报。风险溢价变小也可能推高资产价格，因此“价格上涨”不能单凭一条曲线解释成“企业变好了”。短期中，急着买卖的订单、做市商愿意提供的数量、融资条件也会推动价格。我们做研究时要提出能被数据区分的解释，例如“盈利公告后变化”与“全市场折现率变化”，而不是事后给涨跌配故事。

## 2. 从一根日线走到一次可成交交易

我们账本中的 100 是一个估值输入。要证明真能以它买入，还需要知道它是什么价格。**成交价（trade price）** 是某一笔实际交易的价格；**报价（quote）** 是交易者当前声明愿意买卖的条件；**结算价（settlement price）** 常按交易所规则计算，用于期货每日结算，未必等于最后一笔成交；**估值价（mark）** 是账户记账采用的价格，可以是收盘价、中间价或模型价格。四者各自有用，不能互相替代。

日线把一整天的交易压缩成五个字段，即开盘价、最高价、最低价、收盘价和成交量，常记作 OHLCV。开盘价和收盘价可能来自集合竞价，也可能由数据商选取第一笔、最后一笔有效成交；最高和最低表示区间极值；成交量表示这段时间一共成交多少份。它们没有保留当天发生的先后顺序。若一天最高 105、最低 95，你无法仅凭这根日线知道价格先到 105 还是先到 95，也就不能可靠地判断两个盘中止损止盈订单哪个先触发。




![一根 K 线的四个价格：实体与上下影线](figures/ch01_candle_anatomy.png)

**图 1-2｜K 线解剖。** 日线记录开盘、最高、最低、收盘四价；实体高度为 \(\lvert P_{\mathrm{close}}-P_{\mathrm{open}}\rvert\)。影线表示实体之外的成交价格区间。上涨日仅表示收盘高于开盘。数据类型：概念示意图，数值为教学设定。


回到账本：假如第二天开盘 101、收盘 102，一个前一晚已经发出的买单可能参与开盘交易；一个读完当天收盘才决定买入的模型则不能自动拿到开盘 101。**前视偏差（look-ahead bias）** 指研究在决策时使用了当时尚不可获得的信息。常见错误是用当天完整最高价、最低价、成交量计算信号，又让信号在当天开盘交易。

公开股票指数日线让我们看到真实区间的大小，但它仍没有告诉我们基金 `E` 的盘口、费用和成交约束。尤其是市场剧烈波动的日子，成交量大，只能说明已经发生了很多交易，不能保证你的订单也能低成本成交。




![真实 S&P 500 日线 OHLC 与成交量（2020 年 2–4 月）](figures/ch01_real_sp500_bars.png)

**图 1-3｜真实日线长什么样。** 日线与成交量来自公开 S&P 500 本地快照，2020-03-16 的日对数收益约为 -12.8%。高成交量并不自动意味着低交易成本；本图没有盘口数据，不能直接测得价差或市场冲击。数据类型：公开 OHLCV 快照，2020-02-24 至 2020-04-03。


### 2.1 买价、卖价和“价格没动也亏钱”

假设 `E` 当前最高买入报价为 99.9，最低卖出报价为 100.1。**买一价（bid）** 是别人愿意买入的最高报价，所以我们立即卖出会打到它；**卖一价（ask）** 是别人愿意卖出的最低报价，所以我们立即买入会付出它。**中间价（mid）** 是两者平均数 100；**买卖价差（spread）** 是 \(100.1-99.9=0.2\) 元/份，相对价差是 \(0.2/100=0.2\%\)，也就是 20 个基点。一个**基点（basis point，bp）** 等于 0.01 个百分点，即比例数 0.0001。

我们原先说买 6 份花 600 元，隐含的是“按估值价无摩擦成交”。若真实买入要支付卖一价，现金会变为 \(1000-6\times100.1=399.4\)，持仓仍按中间价估值 600，账户立刻记为 999.4。若立刻全部卖回买一价，再损失 0.6，最后剩 998.8。市场方向没有变，来回交易也损失 1.2 元。

**市价单（market order）** 优先追求尽快成交，但不保证成交价；**限价单（limit order）** 限定可以接受的最差价格，但不保证成交。例如挂 100 元买入，也许一直没有人愿意卖给你。报价档位还包含数量：若卖一只有 2 份，我们买 6 份就可能继续吃掉更高价格的档位。这种随订单大小恶化的成交价格是**市场冲击（market impact）** 的一个来源；从决策价格到实际成交价的差通常称为**滑点（slippage）**，其中也可能混有延迟期间的市场变化。




![五档买卖报价阶梯与中间价、价差](figures/ch01_quote_ladder.png)

**图 1-4｜报价阶梯与半个价差的成本。** 图把 bid/ask 画成两侧的深度条：最好的买价 99.9、最好的卖价 100.1，中间价 100.00，报价价差 0.20。如果你立即买入再立即卖出，价格没变也会损失 0.20，这就是短周期策略最先遇到的成本。数据类型：概念示意图（手写报价；真实盘口每毫秒都在变化）。


In [ ]:
mid, bid, ask, quantity = 100.0, 99.9, 100.1, 6.0
execution_example = pd.Series({
    "cash_after_buy": 1000 - quantity * ask,
    "value_marked_at_mid": 1000 - quantity * ask + quantity * mid,
    "cash_after_round_trip": 1000 - quantity * ask + quantity * bid,
    "relative_spread_bp": (ask - bid) / mid * 10_000,
})
execution_example


这一小段计算也限定了账本的解释范围。后面的公司行动例子仍采用 100 元无费用买入，以便隔离分红和拆股的影响；把报价例子接回去，只需改变买入现金和费用，合同持仓本身不变。实务中应把信号收益与成交损失分列，否则模型表现差时无法判断是预测错了还是实现成本过高。

## 3. 股票和基金：数量与现金为什么会变

### 3.1 股东的权利与基金的包装

**普通股（common stock）** 是企业的剩余权益：企业先承担员工、供应商、债务和税收等义务，剩余利润和清算资产才归股东。买一股不是借给企业一笔有固定还款日的钱，因此股东没有普通债券那样的合同本金。收益可能来自现金分红、企业经营扩张和市场对未来的重新估值；回购会减少流通股数，但也动用了企业现金，不能单独当成免费收益。

我们的 `E` 设定为**交易所交易基金（exchange-traded fund，ETF）**。它不是某一家企业，而是持有一篮子资产的基金份额。基金公布的**每份净值（net asset value，NAV）** 是所持资产净值除以基金份数；交易所里的市场价格则是买卖双方成交的价格，两者可能短时偏离。**申购赎回（creation and redemption）** 机制允许合资格参与者以一篮子资产与基金份额交换，有助于约束偏离，但并不保证任何时刻都等于净值。

选择 ETF 是因为它为本章提供了一个容易理解的可交易包装，而不是因为所有 ETF 风险相同。持有现货黄金、黄金期货和金矿股票的三只基金，名称都可能出现“黄金”，实际现金流和风险完全不同。**管理费（management fee）** 会持续侵蚀净值，**跟踪差异（tracking difference）** 是基金累计收益与目标指数累计收益的差，**跟踪误差（tracking error）** 则通常是两者逐期收益差的标准差。平均少赚一点与忽上忽下地偏离，是不同的问题。

**指数（index）** 是按一套权重和调整规则计算的统计对象。价格指数忽略现金股息再投资，总回报指数计入它。你不能直接购买一份 S&P 500 指数点位；可以购买跟踪它的基金、期货或其他合同，随后承担各自的费用、基差和跟踪差异。后面的真实数据实验使用价格指数，因此只能研究其价格变化，不能称为持有 ETF 的完整投资回报。

### 3.2 分红：从证券价值转成现金权利

第三天 `E` 每份分红 1 元。第二天价格 102，第三天除息后的价格设为 101；这一下降只表示其中 1 元价值从证券里移出。持有 6 份的账户由“606 元证券”加“6 元应收分红”组成，与前一天的 612 元证券价值相同。

**除息日（ex-dividend date）** 决定当天开始买入的交易是否带这次分红权利；**支付日（payment date）** 是现金真正到账的日期，两者通常不同。账本在除息日确认**应收分红（dividend receivable）**，这样不会误报亏损；但这笔应收款在支付前不能自动当成可下单现金。本例约定第五天统一支付 6 元，所以第三、四天现金仍是 400，应收款为 6，第五天现金变为 406、应收款清零。

若总价值记为 \(V_t=B_t+q_tP_t+A_t\)，新出现的 \(A_t\) 就是应收款。实际生产账本还可能需要应付费用、融资利息和待结算资金；这里增加一列的原因很具体：我们必须同时回答“总财富有没有少”和“今天能动用多少现金”。

### 3.3 拆股：尺子变了，财富没有变

第四天发生 1 拆 2 的**股票分割（stock split）**：每份旧份额换成 2 份新份额。份数由 6 变 12，报价由 101 变为 50.5，市值仍是 \(12\times50.5=606\)。如果固定拿 6 份去乘新价格，账本会凭空丢失一半资产。

**公司行动（corporate action）** 包括分红、拆合股、配股、增发、并购和退市等会改变股东权利或证券身份的事件。配股还涉及是否缴款认购；退市可能产生最后可得现金或无法及时变现的权利，不是把最后一个收盘价无限向前填充。即使只是指数成分更新，研究也要区分当时真实存在的成分股与今天仍活着的公司。




![同一证券的原始收盘价与复权收盘价在拆股和分红后分叉](figures/ch01_raw_vs_adjusted.png)

**图 1-5｜raw close 与 adjusted close 的分叉。** 原始成交价在 2:1 拆股日减半，除息日扣除现金股息；向前调整历史价格后，复权曲线去掉这两种机械跳变。直接使用原始价格会把拆股误计为约 50% 的亏损；复权价格也不是当时的成交价。数据类型：固定 seed 合成价格，公司行动为教学设定。


**复权价格（adjusted price）** 是为某种收益计算目的调整历史价格的序列。它可把拆股的机械跳变消掉，也可能按数据商约定计入现金分红。不同供应商对“前复权”“后复权”和分红再投资的定义并不总一致，必须检查文档和实际样例。

在我们的五天里，完整账本从 1000 到 1030，总收益 3%。原始收盘价却从 100 到 52，看起来跌了 48%。两者都能正确描述自己的对象：一个是账户财富，一个是当前每份证券的报价；错误发生在拿后者冒充前者。复权序列适合在已明确口径后计算历史收益，原始价格适合重建当时成交和持仓数量，二者不能随意混搭。


In [ ]:
def hold_account(event_table, initial_cash=1000.0, initial_quantity=6.0):
    """单只现货资产的买入持有账本；分红支付额按账户总额给定。"""
    first_price = float(event_table.iloc[0]["raw_close"])
    cash = initial_cash - initial_quantity * first_price
    quantity, receivable = initial_quantity, 0.0
    records = []
    for day, row in event_table.iterrows():
        # 本例拆股与除息不在同一天；现实需遵循具体事件先后和权利口径。
        quantity *= row["split_ratio"]
        receivable += quantity * row["dividend_per_share"]
        cash += row["dividend_paid"]
        receivable -= row["dividend_paid"]
        market_value = quantity * row["raw_close"]
        records.append({"date": day, "quantity": quantity, "cash": cash,
                        "receivable": receivable, "market_value": market_value,
                        "account_value": cash + receivable + market_value})
    return pd.DataFrame(records).set_index("date")

ledger = hold_account(events)
ledger["account_return"] = ledger.account_value.pct_change(fill_method=None)
ledger


读表时逐行核对：第二天账户 1012；除息日仍是 1012；拆股日仍是 1012；第五天市场价格从 50.5 升到 52，12 份增加 18 元，账户到 1030。到账分红只是资产内部换位置，不会在第五天再次产生收益。这个不变量比“代码运行成功”更有价值：公司行动如果不伴随额外市场变化，不应凭空制造财富。

本例没有自动把分红再投入。若要研究总回报再投资，应在分红现金可用时明确发出买单并计入份数、成交价和费用，不能既用含分红复权收益，又在现金里再加一次分红。税费、零碎股和公司行动通知修订也都未建模，适用范围因此只是理解账本，而不是实际清算。

## 4. 把股票换成债券：不变的账本，不同的现金流

假设现在把 600 元基金换成债券。债券通常规定到期还本和定期付息，**面值（face value）** 是合同本金，**息票（coupon）** 是按约定支付的利息。设每张面值 100、每年付息 5、两年后到期，我们实际买到的是第一年 5 元、第二年 105 元的现金流。

若市场要求每年回报 \(y\)，全价为

\[
P(y)=\frac{5}{1+y}+\frac{105}{(1+y)^2}.
\]

\(P(y)\) 是今天取得这些现金流的价格，\(y\) 是每期折现收益率，两次幂分别对应距离支付还有一、两期。\(y=5\%\) 时价格 100；\(y=6\%\) 时约 98.17。现金流没变，市场要求的收益提高，就只能通过更低买入价格实现。这是债券价格与收益率反向变化的根本原因。

图中的债券采用另一组参数：五期、每期付息 4。我们不必记住图上的每个数，只需要始终把“合同承诺的现金”和“今天为它支付的价格”分开。




![息票债券的现金流时间线与两种收益率下的价格](figures/ch01_bond_cashflows.png)

**图 1-6｜债券现金流与价格—收益率反向关系。** 面值 100、每期息票 4、共 5 期，最后一期收回本金。相同现金流按 \(y=3\%\)、\(5\%\)、\(8\%\) 折现，价格分别为 104.58、95.67、84.03；未来收入不变，折现率越高现值越低。数据类型：由贴现公式计算的概念示例，非市场报价。


### 4.1 收益率报价不是持有收益

**到期收益率（yield to maturity，YTM）** 是使合同现金流现值等于当前全价的内部收益率。它不是“下一年一定赚这么多”：若提前卖出，价格受新利率影响；若违约，合同现金流未必到账；若要把它解释为持有至到期的复合财富收益，还需要对息票再投资作相应假定。

债券市场常报**净价（clean price）**，交易现金支付还需加从上次付息日至今累积的**应计利息（accrued interest）**，合起来是**全价（dirty price）**。例如净价 99、应计利息 1.2，实际需要支付 100.2。把净价直接塞进现金账本，会少扣资金；付息日应计利息回落，又可能被误报成价格损失。这个问题与 `E` 的应收分红相似：价值在不同会计位置之间转移。

利率风险可用**修正久期（modified duration）** 近似量化。对价格函数 \(P(y)\)，定义 \(D_{\rm mod}=-P'(y)/P(y)\)，小变化下有 \(\Delta P/P\approx-D_{\rm mod}\Delta y\)。\(\Delta y\) 要写成比例数：上升 100 个基点是 0.01；若久期是 7，价格约跌 7%。**麦考利久期（Macaulay duration）** 是用各期现金流现值作权重的平均收款时间，按本章每期离散复利口径，它除以 \(1+y\) 得到修正久期。

**凸性（convexity）** 是 \(P''(y)/P(y)\)，在二阶近似中补上 \(\tfrac12 C(\Delta y)^2\)。它解释为什么收益率大幅变化时，单条切线会失准。久期、凸性不是新收入来源，而是帮助我们在账本里估计同一组现金流怎样重估。


In [ ]:
def bond_price(y, coupon=5.0, face=100.0, periods=2):
    t = np.arange(1, periods + 1)
    cashflows = np.full(periods, coupon)
    cashflows[-1] += face
    return float((cashflows / (1 + y) ** t).sum())

y, step = 0.05, 0.0001
price = bond_price(y)
duration = -(bond_price(y + step) - bond_price(y - step)) / (2 * step * price)
convexity = (bond_price(y + step) - 2 * price + bond_price(y - step)) / (step**2 * price)
pd.Series({"price_at_5pct": price, "price_at_6pct": bond_price(0.06),
           "modified_duration": duration, "convexity": convexity,
           "duration_estimated_return": -duration * 0.01,
           "actual_repricing_return": bond_price(0.06) / price - 1})


把这个结果放回 1000 元账户：若用约 600 元持有债券，利率上升导致债券跌 1.83%，对整个账户约贡献 -1.10%，仍要乘持仓比例，而不是直接把收益率变化当收益。这里忽略了持有期息票、时间经过和交易费用，以隔离利率重估。

现实利率不止一个 \(y\)。不同期限的无风险或基准利率构成**收益率曲线（yield curve）**。全期限一起上移是水平变化，长短期限差扩大是斜率变化，中段相对两端变化是曲率变化。一只十年债和一只两年债即使市值相等，对曲线变化也不一样。信用债还承担违约和流动性风险，**信用利差（credit spread）** 是它相对可比期限基准利率的额外收益率；基准利率没变而信用利差扩大，信用债也会跌。只用一条国债收益率序列去解释所有债券盈亏，会漏掉这些渠道。

## 5. 换成外币资产：先统一报告币种

如果 `E` 改为一只欧元计价基金，账本同时需要资产价格和汇率。设用美元报告账户，\(S_t\) 表示每欧元能换多少美元，外币资产价格为 \(P_t^{EUR}\)，美元价值是 \(q_tP_t^{EUR}S_t\)。EUR/USD=1.08 就是这种“美元/欧元”报价；USD/JPY=150 则表示“日元/美元”，方向不能只看数字变大还是变小。

假设买入时欧元基金价格 100 欧元、汇率 1.10 美元/欧元；后来基金涨到 105 欧元，汇率降到 1.00。以欧元看赚了 5%，以美元看价值从 110 到 105，亏约 4.55%。一般有

\[
1+r_t^{USD}=(1+r_t^{EUR})(1+r_t^{FX}),
\]

\(r_t^{EUR}\) 是资产本币收益，\(r_t^{FX}=S_t/S_{t-1}-1\) 是欧元对美元的收益，\(r_t^{USD}\) 是美元报告收益。交叉项在大幅变动时不能忽略。外币现金同样要换算；在不同时间取资产收盘与汇率收盘，也可能拼出实际从未同时存在的账户价值。

**即期（spot）** 约定按近期标准日期交割；**远期（forward）** 约定未来日期交换两种货币；**外汇掉期（FX swap）** 组合近端交换和反方向远端交换，常用于融资。远期汇率不是市场对未来现货的简单平均预测。若美元和欧元同期无风险利率分别为 \(r_{USD},r_{EUR}\)，期限 \(T\) 以年计，在可自由借贷、无交易摩擦条件下，有 \(F=S(1+r_{USD}T)/(1+r_{EUR}T)\)。\(F\) 是美元/欧元远期价，\(S\) 是即期价；它反映两种币种的持有资金成本。真实融资约束会让关系偏离。

所谓**套息（carry）** 是在价格不发生意外变化时，持有资产或融资结构因利息、期限等产生的收益成分，不是无风险保证。高息货币可能在危机时突然贬值。回到账本，外币配置至少要拆成本币资产损益、汇兑损益、利息与对冲成本，不能看到一个美元收益列就直接说模型预测对了股票。


In [ ]:
local_return = 105.0 / 100.0 - 1
fx_return = 1.00 / 1.10 - 1
pd.Series({"asset_return_in_eur": local_return, "eur_return_in_usd": fx_return,
           "combined_usd_return": (1 + local_return) * (1 + fx_return) - 1})


## 6. 换成期货：保证金不是全部风险

假设账户不买现货基金，而买一张教学期货。**期货（futures）** 是交易所标准化的合同，明确标的、到期日、数量单位、交割或现金结算规则。若报价 100 点，**合约乘数（contract multiplier）** 为每点 10 元，一张合约的名义规模是 1000 元；交易所也许只要求 100 元保证金。这 100 元是履约担保，并不是合同只会带来 100 元风险。

价格从 100 点到 99 点，一张多头损失 \((99-100)\times10=-10\) 元。对 1000 元账户是 -1%，对 100 元初始保证金却是 -10%。因此用保证金作分母计算“收益率”，再和全额出资股票比较，会人为放大数字。正确的研究至少并列名义规模、账户资本和保证金占用。

**逐日盯市（mark to market）** 指按结算规则把每日盈亏转成现金收付。**初始保证金（initial margin）** 是开仓所需担保，**维持保证金（maintenance margin）** 是持续持仓的最低要求。价格不利、波动上升或交易所提高比例时，账户可能收到追加保证金要求。即使长期方向最终判断正确，也可能因中途现金不足而被迫平仓。我们的现货账本在买入时扣去全部购买价；期货账本则通常只划拨保证金并逐日收付损益，这两种现金逻辑不能共用一个“花了多少钱”的公式。

### 6.1 换月为什么会影响长期持有

期货有到期日，长期策略必须从即将到期的合约换到较远月份，称为**展期（roll）**。同一时点不同到期月份的报价构成**期限结构（term structure）**。近月 100、远月 102 是向上的结构，常称正向市场或升水结构（contango）；远月 98 是向下的结构，常称逆向市场或贴水结构（backwardation）。

关键是按合约分别记账。若我们把一张近月 100 卖掉，再开一张远月 102，不会仅因为两个报价差 2 点就立即亏 20 元：旧合约按旧账结清，新合约从 102 建仓，期货也不是支付 1020 元买下现货。损益要来自各自持有期间的价格变化及真实费用。

为了隔离期限结构，设现货一直为 100，远月合约买入时 102，临近到期也收敛到 100。那么持有它的多头确实会亏 2 点；这个逐渐发生的**收敛收益** 才是负展期效应的一个清楚例子。如果整个曲线或现货同时变化，实际结果就不能只用开仓时价差判断。




![contango 与 backwardation 两条期货曲线及展期收益方向](figures/ch01_term_structure.png)

**图 1-7｜期限结构决定展期收益的符号。** 两条曲线从同一现货价 100 出发。若现货与期限曲线保持不变，合约随到期向现货收敛，则 contango 对应负的收敛收益，backwardation 对应正的收敛收益。换仓时的价差本身不是立即发生的损益；曲线变化也可能改变实际结果。因此“看对现货方向”和“期货头寸赚钱”是两件事。数据类型：概念示意图（曲线点位为示例）。


In [ ]:
multiplier = 10.0
futures_account = pd.DataFrame({
    "event": ["卖出旧合约", "买入新合约", "新合约到期结算"],
    "contract": ["near", "next", "next"],
    "contract_price": [100.0, 102.0, 100.0],
    "pnl_from_this_step": [0.0, 0.0, (100.0 - 102.0) * multiplier],
})
futures_account


这里旧合约损益设为 0，表示从它自己的成本价 100 退出；不是声称任何旧合约卖出都没有损益。新合约建仓设为零损益，仍忽略手续费与报价价差。若把连续价格表中的 `100,102,100` 直接做百分比收益，就会先虚构一次上涨，再虚构一次下跌，经济路径与账户现金完全不同。

**连续合约（continuous futures series）** 是把多个到期合约按规则拼成的研究序列，本身不是永不过期的可交易合同。直接拼接会产生换月跳点，价格加减调整或比例调整可以去掉某些跳点，却未必保留实际可交易损益。做严谨回测要知道每日持有哪个合同、何时换月、每张乘数是多少、实际结算变动与费用是什么。

商品期货回报还可能涉及现货变化、合约随到期收敛、抵押现金利息等成分。常见“现货收益加展期收益加抵押品收益”的表达是一种归因近似，其定义取决于指数与资金配置规则。对于我们的 1000 元账户，最稳妥的起点仍是逐张合同计算现金损益，再把剩余现金利息加回，而不是先相信某条连续价格线。

期货的**成交量（volume）** 是某时间段内成交的张数，**未平仓量（open interest）** 是尚未了结的合同数。一位旧持有人把合约转给新持有人，成交量增加，未平仓量可以不变；买卖双方都新开仓时，未平仓量才增加。二者可用于研究交易活跃度，却不是可互换的“热度”标签，更不直接等于可容纳我们订单的盘口深度。

## 7. 换成期权：不能只预测方向

如果账户想保留上涨机会，同时把某个头寸的损失限制在已支付金额，可以买入**期权（option）**。看涨期权（call）给买方以**执行价（strike）** \(K\) 买入标的的权利；看跌期权（put）给买方以 \(K\) 卖出的权利。买方支付**权利金（premium）**，卖方收取权利金并承担对应义务。实际合同还要明确欧式或美式行权、现金或实物交割、乘数和到期日期。

先看一份欧洲式看跌期权，执行价 100、权利金 4，到期标的价格 \(S_T\) 为 80。到期支付为 \(\max(100-80,0)=20\)，扣除权利金后净利润 16；若价格为 105，支付为零，净亏 4。一般到期支付为

\[
H_T^{call}=\max(S_T-K,0),\qquad H_T^{put}=\max(K-S_T,0).
\]

\(H_T\) 是到期合同支付，\(S_T\) 是到期标的价格，\(K\) 是执行价，\(T\) 是到期时点。支付不是净利润：还要扣去开始支付的权利金及资金成本，并乘合约数量和乘数。账户若用 600 元支付权利金，最多亏掉这 600 元，不代表它对标的价格只有 600 元敏感度。




![四种基本期权头寸的到期 payoff 与含权利金净利润](figures/ch01_option_payoffs.png)

**图 1-8｜期权到期 payoff 与净利润。** 四个面板分别为买入和卖出看涨、看跌；灰线是到期 payoff，蓝色虚线是计入权利金的净利润。买方最大损失为权利金；未对冲看涨卖方亏损可无上限，看跌卖方的亏损上限则受执行价约束。数据类型：概念示例，\(K=100\)，看涨、看跌权利金分别为 6 和 4。


把期权接到账本还有一个容易忽略的地方：到期之前，不能只用到期支付估值。例如标的 100、执行价 100 的看涨期权，当下立即行权价值为零，但到期前仍可能上涨，所以市场价格可能为 6。把它每天都估成零，会把真实持有价值和价格波动删掉。

**隐含波动率（implied volatility）** 是将市场期权价格代入一个定价模型后反解出的波动率参数，它反映该模型下市场为不确定性支付了多少价格，不等于已实现波动，也不是对未来波动的无条件承诺。同样价格方向判断正确，若买入时期权很贵、随后隐含波动率大幅下降，期权持有者仍可能亏钱。

局部重估可写成

\[
\Delta C\approx\Delta\,\Delta S+\tfrac12\Gamma(\Delta S)^2
+\mathrm{Vega}\,\Delta\sigma+\Theta\,\Delta t+\mathrm{Rho}\,\Delta r.
\]

\(C\) 是期权价格，\(S\) 是标的价格，\(\sigma\) 是模型波动率参数，\(t\) 是时间，\(r\) 是利率；左侧是期权价格变化。右侧的 Delta 是一阶价格敏感度，Gamma 是价格敏感度随标的变化的程度，Vega 是波动率敏感度，Theta 是时间流逝影响，Rho 是利率敏感度。各市场对 Vega 的“一个单位”可能按波动率 1.00 或一个百分点报告，必须明确单位。

比如某时点 Delta=0.5，标的涨 1 元时，期权只因方向变化约涨 0.5 元；但若 Vega 项和时间项合计 -0.8 元，总变化仍可能为 -0.3。这个近似只在变动不大且定价模型适用时有帮助，接近到期、跳空或深度虚值时应重新定价。不同到期日、执行价的隐含波动率组成**波动率曲面（volatility surface）**，所以股票上的单个“涨跌预测”不能自动变成完整的期权研究。

如果卖出期权，权利金不能全部当作立刻实现的可花利润：卖方仍有待履行的负债。未对冲看涨期权的亏损可随标的上涨无限扩大，看跌期权的亏损虽有价格归零带来的上界，却仍可能远大于收取的权利金。原先 \(B+qP\) 的账户表达在这里必须包括负的合同价值和保证金约束。

## 8. 同一本账本还需要哪些时间和规则

### 8.1 信息的日期与交易的日期

假设 `E` 的某成分公司在 3 月 31 日结束季度，4 月 26 日晚披露利润。报告描述的是第一季度，但投资者直到 4 月 26 日晚才知道结果。把利润数据按“季度末 3 月 31 日”连接到价格表，再从 4 月 1 日买入，就是让账本使用未来信息。

至少要区分**经济所属期（period end）**、**披露时间（release time）**、数据库可取得时间和订单成交时间。数据商后来修正了历史财报，也不意味着过去的投资者已经看到修正值。**时点一致数据（point-in-time data）** 记录在各个历史决策时刻真正已知的版本；它对财报、指数成分、分析师预期和另类数据都很重要。

相似地，收盘产生信号、下一次开盘成交，持有收益应从那次真实成交之后开始。假如使用跨市场信息，还要带时区：美国周一收盘信息在亚洲周二早间才可见，不能按同一个自然日期连接之后让它影响亚洲周一交易。日历是否开市、临时停牌、涨跌停和最小交易单位，也决定了目标持仓能否实现。

### 8.2 加密资产和另类数据没有绕过账本

加密资产经常全天候交易，但不存在唯一的全球收盘价。你要指定交易所、计价币和每日切分时点。以某稳定币计价的价格还承担稳定币相对报告币种变化、托管和提取限制的风险。两个交易所显示价差，并不保证资产能及时转过去完成套利。

加密**永续合约（perpetual swap）** 没有常规到期日，常通过**资金费率（funding rate）** 在多空之间定期支付费用，使合同价格受到现货锚定约束。它与普通期货到期收敛的机制不同。多头即使看对上涨，也可能因高额资金支付、杠杆清算或平台事件亏损；账本应记录资金费发生时间和付费方向。

**另类数据（alternative data）** 可能是新闻、招聘、支付、网页或卫星观测。研究价值来自它是否提供新的、当时可得的信息，而不是数据名字是否新颖。例如网页抓取表中的“2020 年发布日期”可能是 2024 年回填的页面，不能证明 2020 年就能取得其内容。样本覆盖变化、删除和合规使用范围都应进入数据定义；一个漂亮特征不能替代这些事实。

这些限制最终都回到原来的账户：今天知道什么，今天能否下单，未来哪一笔现金会进入或离开，发生失败时损失由谁承担。只要这条线索明确，新增资产和数据就不必靠死记名词来理解。


In [ ]:
# 用显式数据字典约束后续研究：指标与可交易资产分开。
contracts = pd.DataFrame([
    {"id": "E", "kind": "现货基金", "quote_unit": "元/份", "multiplier": 1,
     "return_rule": "原价+数量+现金流", "tradable": True},
    {"id": "bond_yield", "kind": "收益率指标", "quote_unit": "年化比例", "multiplier": np.nan,
     "return_rule": "先映射到具体债券重估", "tradable": False},
    {"id": "future_next", "kind": "指定到期月期货", "quote_unit": "点", "multiplier": 10,
     "return_rule": "价格差×乘数+现金利息", "tradable": True},
    {"id": "VIX", "kind": "隐含波动率指标", "quote_unit": "波动率点", "multiplier": np.nan,
     "return_rule": "不可直接作为现货持有收益", "tradable": False},
])
contracts


## 9. 用本地公开数据检验“对象、单位、时间”

现在把同一个账本思路拿到真实数据，但先停在输入定义这一层。本地快照包括 S&P 500 价格指数、十年期美国国债收益率报价和 VIX。我们不把它们假装成三只基金，而是做三类各自有意义的变动：指数价格的简单收益、收益率的基点变化、VIX 的点数变化。

对于本地这份 `^TNX` 快照，文件数值按收益率百分数表示，例如 4.5 表示年化 4.5%。因而从 4.5 到 4.6 是上升 0.1 个百分点，即 10 个基点；它不是一只债券涨了 \(4.6/4.5-1\)。这个转换只适用于已经核实的本地文件口径，换供应商时需要重新核对，而不能把某个缩放规则当成所有利率数据的常识。

真实日期连接仍只表示观察日相同，不证明三个市场在同一瞬间可交易。若日历不完全一致，应该保留缺失值并调查原因。下面先在各自序列上计算变化，再按日期合并，避免其他市场多出一个开市日就切断股票连续两次收盘的收益。我们只显示表格和图，不保存新图或覆盖教材插图。


In [ ]:
import matplotlib.pyplot as plt

for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidates = [base, base / "research", base / "run-quant" / "research"]
    found = [p for p in candidates if (p / "sources/external/data/yahoo_sp500.csv").exists()]
    if found:
        research_root = found[0]
        break
else:
    raise FileNotFoundError("未找到 research/sources/external/data 本地快照")

data_dir = research_root / "sources/external/data"
spx = pd.read_csv(data_dir / "yahoo_sp500.csv", parse_dates=["date"]).set_index("date")["close"].sort_index()
tnx = pd.read_csv(data_dir / "yahoo_tnx.csv", parse_dates=["date"]).set_index("date")["close"].sort_index()
vix = pd.read_csv(data_dir / "vix_daily.csv", parse_dates=["date"]).set_index("date")["vix_close"].sort_index()
market_changes = pd.concat({
    "spx_price_return": spx.pct_change(fill_method=None),
    "yield_change_bp": tnx.diff() * 100,
    "vix_change_points": vix.diff(),
}, axis=1, sort=True)
market_changes.loc["2020-03-09":"2020-03-20"].round(4)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
spx.loc["2018":"2025"].plot(ax=axes[0], title="S&P 500 price index", color="#2563eb")
tnx.loc["2018":"2025"].plot(ax=axes[1], title="10Y yield, percent", color="#64748b")
vix.loc["2018":"2025"].plot(ax=axes[2], title="VIX, volatility points", color="#dc2626")
fig.tight_layout()
plt.show()


观察 2020 年某个市场剧烈变化的日期，逐列说出它对账户的含义。股票指数下跌暗示股票方向敞口可能亏损；国债收益率下降可能令固定现金流债券上涨，但要知道久期、期限和信用风险才能计算；VIX 上升可能让部分期权变贵，但不能把它的百分比变化直接记作“持有 VIX 的收益”。如果无法给某一列写出从观察到合同损益的映射，应先保留为解释变量，而不是强行放入资产收益矩阵。

## 10. 常见误解：用账本逐个拆开

**“价格就是可以成交的价格。”** 中间价能帮助统一估值，却没有保证有人愿意以它成交。我们的 6 份买入在卖一价成交，已经让账户损失半个价差；订单更大还要看深度。研究应分别记录决策价、报价、成交价和估值价。

**“复权后就不需要公司行动信息。”** 复权收益可以简化特定持有回报计算，却不能独自回答当天有多少真实股份、多少可用现金、谁有分红资格。我们的拆股和分红账本需要事件和支付日，不能只保留一条平滑价格线。

**“保证金小，所以期货风险小。”** 保证金是担保需求，经济损益来自合约乘数乘价格变化。100 元担保支持 1000 元名义规模时，小价格变化也会吃掉很多可用现金。仓位规模首先应对照账户资本和可承受损失。

**“看对方向就能赚到钱。”** 外币汇率可能抵消资产上涨，债券信用利差可能抵消基准利率下降，期权波动率和时间损耗可能抵消标的上涨。把这些变化分别接入同一账本，才能说明究竟预测了什么。

**“日线数据天然没有时间问题。”** 当天收盘价、最终成交量、晚间公告都在不同时间形成。低频只减少了记录数量，并不会自动消除前视偏差或跨时区错误。

## 11. 手工练习

### 练习 A：补完五天账户

不运行代码，写出本章五天的持仓数量、现金、应收分红、证券市值和账户价值。再解释：若第三天就将应收分红视为可用现金，财富计算可能仍然正确，交易模拟为什么会错？

<details><summary>参考答案</summary>

五天数量为 6、6、6、12、12；现金为 400、400、400、400、406；应收分红为 0、0、6、6、0；证券市值为 600、612、606、606、624；账户价值为 1000、1012、1012、1012、1030。第三天只有分红权利，尚未收到现金；若用这 6 元加仓，就模拟了尚无资金支持的订单，除非额外建模融资及其成本。

</details>

### 练习 B：同一个“上涨”

某日股票价格指数上涨 1%，十年债收益率由 4% 升到 4.1%，EUR/USD 由 1.10 升到 1.11。用美元报告的账户同时持有股票基金、修正久期为 7 的美元债和欧元现金。分别说明大致损益方向；债券忽略息票和凸性。

<details><summary>参考答案</summary>

股票基金一般受益，但具体涨幅还受跟踪差异和持仓影响。债券收益率上升 10 个基点，即 0.001，近似价格收益为 \(-7\times0.001=-0.7\%\)。欧元现金以美元计价升值约 \(1.11/1.10-1=0.909\%\)。三列都“上升”，对应账户方向却不同。

</details>

### 练习 C：展期不是两份价格之差

你在 100 点建立近月多头，之后在 103 点平仓，同时在 105 点建立远月多头，最后远月在 104 点平仓。各一张，乘数每点 10 元，不计成本。写出两张合同各自损益、总损益，解释为什么不能把 103 到 105 记为立即亏损。

<details><summary>参考答案</summary>

近月赚 \((103-100)\times10=30\) 元；远月亏 \((104-105)\times10=-10\) 元；合计赚 20 元。105 是新合同的开仓基准，不是旧合约持有期间的价格变化；换月并没有按两个合同价格全额相减支付现金。实际交易还要扣两个方向的费用和价差。

</details>

### 练习 D：研究者应在何时买入

某信号要用美国周一收盘后发布的公司公告。研究表把它标成周一，却在亚洲周一收盘价买入相关资产。怎样修改时间字段和成交假设？

<details><summary>参考答案</summary>

保留带时区的公告发布时间与研究系统实际收到时间，选择之后相关市场第一个可交易时刻，并考虑处理延迟和订单提交规则。通常最早会落在亚洲周二或之后，具体还取决于假日和交易时段。不能只把日期加一天而忽略跨市场日历。

</details>

## 12. 自测与思考

1. 为什么相同的价格百分比变化，未必对应相同的账户收益？
2. 一份字段字典至少要补充哪些事实，才能把一列数据放进交易研究？
3. 如果只允许对本章账本增加一项现实因素，你会先增加成交费用、融资还是公司行动税费？如何说明选择？
4. 将 `E` 换成期权时，为什么 `cash + quantity × option_price` 仍可作估值表达，却不能沿用现货价格的预测模型就宣称完成了研究？

<details><summary>参考答案</summary>

1. 账户收益还取决于投入比例、现金、汇率、杠杆、分红、费用和合同规则。只有这些都匹配时，资产收益才会按明确权重传到账户。
2. 包括对象身份、报价单位、币种、数据来源、交易日历、字段生成和可见时间、价格类型、合同乘数、公司行动或到期规则，以及能否直接交易。并非所有字段都需要填相同内容，但不能省略经济解释。
3. 现货低频教学账户可先加成交费用，因为它直接修正买卖现金，易于隔离解释；高杠杆期货可能先加融资与保证金约束；高股息账户则税费也重要。答案依赖研究用途，关键是说清新增机制改变哪项现金流和哪个结论。
4. 估值恒等式仍成立，但期权价格受到执行价、到期日、隐含波动率、非线性和买卖报价影响。方向预测只覆盖其中一个渠道，还缺合同选择、定价、风险和可实现成交。

</details>

## 13. 本章小结

我们从“1000 元买 6 份基金”得到了一条可以手算、可以运行的主线。报价决定交易付出什么，原始价格决定当时每份证券如何估值，公司行动决定份数和应收现金，报告币种决定怎样汇总，合同规则决定损益何时变成现金。股票、债券、外汇、期货和期权改变的是这些映射，而不是给价格表换个名字。

研究的第一份可靠产物不是漂亮的收益曲线，而是一份能逐行解释的钱和权利的账本。下一章沿用这种记账意识，把逐期损益变成收益、财富路径和风险报告，讨论一个看似赚钱的结果是否值得继续研究。
